# 第14章　利率期货与远期

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch14_futures_forwards.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch14_futures_forwards.ipynb)

复现例14.1（转换因子）、例14.2（CTD）、例14.3 + 图14-1（久期中性套保）、例14.4（FRA），并用 QuantLib 取远期利率。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import futures as fut
from fi import plotting
plotting.use_chinese_style()


## 例14.1　转换因子（名义票息 3%）


In [ ]:
for c, yrs in [(0.028, 7), (0.032, 8), (0.030, 9)]:
    print(f'票息{c*100:.1f}%, {yrs}年: CF = {fut.conversion_factor(c, yrs):.4f}')


## 例14.2　确定 CTD（期货价 100）


In [ ]:
F = 100.0
bonds = [
    {'name': '券A(2.8%,7y)', 'clean_price': 99.10, 'conversion_factor': fut.conversion_factor(0.028, 7)},
    {'name': '券B(3.2%,8y)', 'clean_price': 101.55, 'conversion_factor': fut.conversion_factor(0.032, 8)},
    {'name': '券C(3.0%,9y)', 'clean_price': 100.30, 'conversion_factor': fut.conversion_factor(0.030, 9)},
]
best, table = fut.ctd(bonds, F)
for r in table:
    print(f"{r['name']}: CF={r['conversion_factor']:.4f}  净价={r['clean_price']}  毛基差={r['gross_basis']:.4f}")
print('-> CTD =', best['name'])


## 例14.3 + 图14-1　久期中性套保与回测


In [ ]:
port_dv01 = 5 * 1e8 * 1e-4                       # 组合 DV01（元/bp）
ctd_dv01 = fut.bond_dv01(0.032, 8, 0.032)        # CTD 每百元 DV01
ctd_cf = best['conversion_factor']
contract_dv01 = fut.futures_dv01(ctd_dv01, ctd_cf) / 100 * 1e6   # 每张合约 DV01（元/bp）
N = -port_dv01 / contract_dv01
print(f'组合 DV01 = {port_dv01:.0f} 元/bp')
print(f'每张期货 DV01 = {contract_dv01:.2f} 元/bp')
print(f'套保手数 N = {N:.1f}（卖出约 {abs(round(N))} 张）')

shocks = np.linspace(-100, 100, 41)
net_dv01 = port_dv01 + round(N) * contract_dv01
fig, ax = plotting.new_axes()
ax.plot(shocks, -port_dv01*shocks/1e4, label='未对冲组合')
ax.plot(shocks, -net_dv01*shocks/1e4, label=f'对冲后（卖出 {abs(round(N))} 张）')
ax.axhline(0, ls=':', color='gray')
ax.set_xlabel('利率平行冲击 (bp)'); ax.set_ylabel('组合损益（万元）')
ax.set_title('图14-1　国债期货久期中性套保的效果'); ax.legend()
fig.tight_layout()


## 例14.4　FRA 估值


In [ ]:
v = fut.fra_value(notional=1e8, contract_rate=0.025, forward_rate=0.028, tau=0.25)
print(f'名义1亿, 合约2.5%, 远期2.8%, 3个月: FRA多头价值 ≈ {v:.0f} 元')


## 14.7　QuantLib：由曲线取远期利率（FRA 公允利率）


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
curve = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.025, dc))
d1, d2 = today + ql.Period(1, ql.Years), today + ql.Period(15, ql.Months)
fwd = curve.forwardRate(d1, d2, dc, ql.Simple).rate()
print(f'QuantLib 1y3m 远期利率 = {fwd*100:.4f}%（FRA 公允约定利率）')


---

> 小结：国债期货以名义券+一篮子可交割券交易，期货价格与久期由 CTD（毛基差最小）决定；
> DV01 中性套保 N = −组合DV01/每张期货DV01，对冲一阶平行风险；FRA 公允利率 = 远期利率。
